In [14]:
import scipy
import shutil
import zipfile
import logging
import datetime
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [15]:
BASE_DIR = Path("AffectiveROAD/AffectiveROAD_Data/Database")
SIGNAL_DIR = BASE_DIR / "E4"
LABEL_DIR = BASE_DIR / "Subj_metric"
TEMP_DIR = BASE_DIR / "temp"
OUTPUT_DIR = Path("data_3")

In [36]:
TEST_SUBJECTS = 4
SAMPLING_RATE = 4
SAMPLING_RATES = {
    'ACC': 32,
    'BVP': 64,
    'EDA': 4,
    'TEMP': 4,
    'label': 4,
}

In [17]:
def process_features_folders(base_directory, processed_dataset_path):
    
    # Loop through serial numbers from 1 to 13
    for serial_number in tqdm(range(1, 14), desc="Processing folders"):
        folder_name = f'{serial_number}-E4-Drv{serial_number}'
        folder_path = base_directory / folder_name

        # Check if the folder exists
        if folder_path.exists() and folder_path.is_dir():
            
            # List files in the folder
            files_in_folder = list(folder_path.iterdir())

            # Unzip each zip file in the folder
            for file_path in tqdm(files_in_folder, desc="Unzipping files", leave=False):
                if file_path.name.endswith('.zip'):
                    
                    # Create a new folder for each unzipped content
                    new_folder_name = f'{serial_number}_unzipped'
                    new_folder_path = processed_dataset_path / new_folder_name
                    new_folder_path.mkdir(parents=True, exist_ok=True)

                    with zipfile.ZipFile(file_path, 'r') as zip_ref:
                        zip_ref.extractall(new_folder_path)

        else:
            tqdm.write(f"Folder {folder_name} not found.")
    
    logger.info("Feature folders processed and unzipped.")

In [18]:
def process_label_files(base_directory, processed_dataset_path):
    # Loop through serial numbers from 1 to 13
    for serial_number in tqdm(range(1, 14), desc="Processing files"):
        file_name = f'SM_Drv{serial_number}.csv'
        file_path = base_directory / file_name
        # Check if the file exists
        if file_path.exists() and file_path.is_file():

            # New filepath to copy file to
            new_folder_name = f'{serial_number}_unzipped'
            new_folder_path = processed_dataset_path / new_folder_name
            # Copy the file to the new folder
            new_file_path = new_folder_path / file_name
            shutil.copy(file_path, new_file_path)

        else:
            tqdm.write(f"File {file_path} not found.")
    
    logger.info("Label files processed and copied.")

In [19]:
def eda_process(
    eda_signal, sampling_rate=1000, method="neurokit", report=None, **kwargs
):
    # Sanitize input
    eda_signal = nk.signal.signal_sanitize(eda_signal)
    methods = nk.eda.eda_methods.eda_methods(sampling_rate=sampling_rate, method=method, **kwargs)

    # Preprocess
    # Clean signal
    eda_cleaned = eda_signal
    if methods["method_phasic"] is None or methods["method_phasic"].lower() == "none":
        eda_decomposed = pd.DataFrame({"EDA_Phasic": eda_cleaned})
    else:
        eda_decomposed = nk.eda_phasic(
            eda_cleaned,
            sampling_rate=sampling_rate,
            method=methods["method_phasic"],
            **methods["kwargs_phasic"],
        )

    # Find peaks
    peak_signal, info = nk.eda_peaks(
        eda_decomposed["EDA_Phasic"].values,
        sampling_rate=sampling_rate,
        method=methods["method_peaks"],
        amplitude_min=0.1,
        **methods["kwargs_peaks"],
    )
    info["sampling_rate"] = sampling_rate  # Add sampling rate in dict info

    # Store
    signals = pd.DataFrame({"EDA_Raw": eda_signal, "EDA_Clean": eda_cleaned})

    signals = pd.concat([signals, eda_decomposed, peak_signal], axis=1)

    return signals, info

In [20]:
def ppg_process(
    ppg_signal, sampling_rate=1000, method="elgendi", method_quality="templatematch", report=None, **kwargs
):
    # Sanitize input
    ppg_signal = nk.misc.as_vector(ppg_signal)
    methods = nk.ppg.ppg_methods(sampling_rate=sampling_rate, method=method, method_quality=method_quality, **kwargs)

    # Clean signal
    ppg_cleaned = ppg_signal

    # Find peaks
    peaks_signal, info = nk.ppg_peaks(
        ppg_cleaned,
        sampling_rate=sampling_rate,
        method="bishop",
    )

    info["sampling_rate"] = sampling_rate  # Add sampling rate in dict info

    # Rate computation
    rate = nk.signal.signal_rate(
        info["PPG_Peaks"], sampling_rate=sampling_rate, desired_length=len(ppg_cleaned)
    )

    # Assess signal quality
    quality = nk.ppg_quality(
        ppg_cleaned,
        peaks=info["PPG_Peaks"],
        sampling_rate=sampling_rate,
        method=methods["method_quality"],
        **methods["kwargs_quality"]
    )

    # Prepare output
    signals = pd.DataFrame(
        {
            "PPG_Raw": ppg_signal,
            "PPG_Clean": ppg_cleaned,
            "PPG_Rate": rate,
            "PPG_Quality": quality,
            "PPG_Peaks": peaks_signal["PPG_Peaks"].values,
        }
    )

    return signals, info


In [21]:
def process_data(serial_number, base_path):
    folder_name = f"{serial_number}_unzipped"
    current_folder_path = base_path / folder_name
    
    # Check if the folder exists
    if current_folder_path.exists() and current_folder_path.is_dir():
        # Define file paths for EDA, HR_new, TEMP, and STRESS data
        acc_path = current_folder_path / "ACC.csv"
        bvp_path = current_folder_path / "BVP.csv"
        eda_path = current_folder_path / "EDA.csv"
        temp_path = current_folder_path / "TEMP.csv"
        label_path = current_folder_path / f"SM_Drv{serial_number}.csv"

        # Read data frames
        acc = pd.read_csv(acc_path, header=2).values
        bvp = pd.read_csv(bvp_path, header=2).values
        eda = pd.read_csv(eda_path, header=2).values
        temp = pd.read_csv(temp_path, header=2).values
        label = pd.read_csv(label_path, header=1).values
        
        data = {
            'ACC': acc,
            'BVP': bvp,
            'EDA': eda,
            'TEMP': temp,
            'label': label
        }
        
        for key in data.keys():
            if key == 'ACC':
                temp = []
                for i in range(data[key].shape[1]):
                    temp.append(nk.signal_resample(data[key][:, i], sampling_rate=SAMPLING_RATES[key], desired_sampling_rate=SAMPLING_RATE))
                # temp.append(np.linalg.norm(np.array(temp), axis=0))
                # data['ACC'] = np.stack(temp, axis=1)
                data['ACC'] = np.linalg.norm(np.array(temp), axis=0)
            else:
                resampled = nk.signal_resample(data[key][:, 0], sampling_rate=SAMPLING_RATES[key], desired_sampling_rate=SAMPLING_RATE)
                data[key] = resampled
        
        bvp, _ = ppg_process(data['BVP'], sampling_rate=SAMPLING_RATE)
        eda, _ = eda_process(data['EDA'], sampling_rate=SAMPLING_RATE)
        
        #acc = pd.DataFrame(data['ACC'], columns=['ACC_x', 'ACC_y', 'ACC_z', 'ACC_net'])
        acc = pd.DataFrame(data['ACC'], columns=['ACC'])
        bvp = bvp.drop(columns=['PPG_Raw', 'PPG_Clean'])
        eda = eda.drop(columns=['EDA_Raw', 'EDA_Clean'])
        temp = pd.DataFrame(data['TEMP'], columns=['TEMP'])

        label = pd.DataFrame(data['label'], columns=['label'])

        # Determine the minimum length among data frames
        min_len = min(len(acc), len(bvp), len(eda), len(temp), len(label))

        # Take the first min_len rows from each data frame
        acc = acc.iloc[:min_len]
        bvp = bvp.iloc[:min_len]
        eda = eda.iloc[:min_len]
        temp = temp.iloc[:min_len]
        label = label.iloc[:min_len]
        
        # Concatenate the data column-wise
        df = pd.concat([acc, bvp, eda, temp, label], axis=1, join='outer')
        
        df.reset_index(drop=True, inplace=True)

        for col in df.columns:
            if np.array_equal(df[col].unique(), np.array([0., 1.])):
                df[col] = df[col].astype(int)
        
        df = df.assign(
            subject_id=serial_number,
            label=(df['label'] > 0.75).astype(int)
        )
    
        return df

In [22]:
def process_all_data(base_path):
    all_data = {}

    # Loop through folders with names in the format X_unzipped
    for serial_number in tqdm(range(1, 14), desc="Processing Folders"):
        df_subject = process_data(serial_number, base_path)
        all_data[serial_number] = df_subject
    
    logger.info("All CSV files combined")
    
    return all_data

In [34]:
def data_split(all_data, test_subjects: int):
    all_data_list = list(all_data.values())
    n_subjects = len(all_data)
    n_test = test_subjects
    n_train = n_subjects - n_test

    df_train = pd.concat(all_data_list[:n_train], ignore_index=True, axis=0)
    df_test = pd.concat(all_data_list[n_train:], ignore_index=True, axis=0)

    df_train_with_anomaly = df_train.reset_index(drop=True)
    df_train = df_train[df_train['label'] == 0].reset_index(drop=True) # Use only non-stress data for training 
    df_test = df_test.reset_index(drop=True) # Use all data for testing
    
    logger.info('Data split into train and test sets')
    logger.info('Train data shape: %s', df_train.shape)
    logger.info('Test data shape: %s', df_test.shape)

    return df_train, df_train_with_anomaly, df_test

In [24]:
process_features_folders(SIGNAL_DIR, TEMP_DIR)

Processing folders: 100%|██████████| 13/13 [00:00<00:00, 23.16it/s]
INFO:__main__:Feature folders processed and unzipped.


In [25]:
process_label_files(LABEL_DIR, TEMP_DIR)

Processing files: 100%|██████████| 13/13 [00:00<00:00, 2423.48it/s]
INFO:__main__:Label files processed and copied.


In [26]:
all_data = process_all_data(TEMP_DIR)

Processing Folders: 100%|██████████| 13/13 [13:10<00:00, 60.81s/it]
INFO:__main__:All CSV files combined


In [37]:
df_train, df_train_with_anomaly, df_test = data_split(all_data, TEST_SUBJECTS)

INFO:__main__:Data split into train and test sets
INFO:__main__:Train data shape: (48695, 16)
INFO:__main__:Test data shape: (80332, 16)


In [38]:
df_train.ffill(inplace=True)
df_train_with_anomaly.ffill(inplace=True)
df_test.ffill(inplace=True)

In [39]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_train.to_csv(OUTPUT_DIR / 'train.csv', index=False)
df_train_with_anomaly.to_csv(OUTPUT_DIR / 'train_with_anomaly.csv', index=False)
df_test.to_csv(OUTPUT_DIR / 'test.csv', index=False)

In [40]:
df_train.label.value_counts()

label
0    48695
Name: count, dtype: int64

In [41]:
df_train_with_anomaly.label.value_counts()

label
1    65528
0    48695
Name: count, dtype: int64

In [42]:
df_test.label.value_counts()

label
0    59964
1    20368
Name: count, dtype: int64

In [43]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48695 entries, 0 to 48694
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ACC               48695 non-null  float64
 1   PPG_Rate          48695 non-null  float64
 2   PPG_Quality       48695 non-null  float64
 3   PPG_Peaks         48695 non-null  int64  
 4   EDA_Tonic         48695 non-null  float64
 5   EDA_Phasic        48695 non-null  float64
 6   SCR_Onsets        48695 non-null  int64  
 7   SCR_Peaks         48695 non-null  int64  
 8   SCR_Height        48695 non-null  float64
 9   SCR_Amplitude     48695 non-null  float64
 10  SCR_RiseTime      48695 non-null  float64
 11  SCR_Recovery      48695 non-null  int64  
 12  SCR_RecoveryTime  48695 non-null  float64
 13  TEMP              48695 non-null  float64
 14  label             48695 non-null  int64  
 15  subject_id        48695 non-null  int64  
dtypes: float64(10), int64(6)
memory usage: 5